In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# Use accelerator (GPU/MPS) if available, otherwise fall back to CPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [ ]:
# build neural network
# 신경망 모델의 역할 : 특징을 집어넣으면 모델은 자신의 고유한 가중치로 각 층(선형 변환 층, 비선형 변환 층)을 통과하면서 예측을 시도한다.
# 각 layer들은 가중치를 기반으로 계산하기 때문에 가중치가 이상하면 틀린 결과물을 리턴한다.
# 예) 각 layer들이 가중치와 특징을 받았을 때 가중치가 이상하다면 오해를 하여 중요하지도 않은 것들을 중요하다고 생각하고 정작 중요한 것을 무시하게 된다.
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(28 * 28, 512),
      nn.ReLU(),
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10)
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits


In [ ]:
# move model to the selected accelerator
# model 객체가 가중치를 계속 들고 다님
model = NeuralNetwork().to(device)

In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
print(logits.shape)
# softmax는 raw 데이터(logit)의 값을 0~1 사이의 확률로 정규화해주는 클래스
# dim이 1이기 때문에 1번째 축 방향으로 확률을 계산한다.
pred_probab = nn.Softmax(dim=1)(logits)
# 마찬가지로 정규화된 값의 argmax는 1번째 축 방향에서 제일 큰 값의 인덱스를 반환한다.
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

torch.Size([1, 10])
Predicted class: tensor([8], device='cuda:0')


In [ ]:
input_image = torch.rand(3, 28, 28)

# flatten 클래스는 0번째 축을 제외한 뒤 나머지 축들을 1차원으로 평탄화해주어 컴퓨터가 읽게 해준다.
# 0번째 축은 평탄화하지 않는 것이 default로 설정되어 있다.
flatten = nn.Flatten()
flat_image = flatten(input_image)

# 픽셀값에 가중치를 곱하고 편향을 더해 압축하는 선형 변형 진행
# 가중치 : 특징이 얼마나 중요한지 결정하는 값(무엇이 정답인지를 예측하는 데 핵심 역할)
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
hidden1 = layer1(flat_image)

# 선형적 변형만 하면 복잡한 패턴은 파악하기 힘들기 때문에 비선형적인 변형도 도입한다.
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1799,  0.4032,  0.1576,  0.6505, -0.1150, -0.0208, -0.1037,  0.0734,
         -0.1329,  0.2437,  0.0028, -0.1848,  0.0685, -0.5188,  0.0406,  0.5040,
          0.8946,  0.2383, -0.0073,  0.6011],
        [-0.3432,  0.5176, -0.3421,  0.2296, -0.2156,  0.1886,  0.1500, -0.0305,
          0.3142, -0.2216, -0.1604, -0.1538, -0.4564, -0.2004,  0.4046,  0.2018,
          0.7139,  0.2081,  0.1677,  0.5164],
        [-0.2070,  0.6218, -0.2399,  0.5136, -0.2651,  0.0326, -0.1548,  0.4725,
          0.2423,  0.2473, -0.5754, -0.2782,  0.0443, -0.1948,  0.0787,  0.7869,
          0.8762, -0.0217, -0.0975,  0.6533]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.4032, 0.1576, 0.6505, 0.0000, 0.0000, 0.0000, 0.0734, 0.0000,
         0.2437, 0.0028, 0.0000, 0.0685, 0.0000, 0.0406, 0.5040, 0.8946, 0.2383,
         0.0000, 0.6011],
        [0.0000, 0.5176, 0.0000, 0.2296, 0.0000, 0.1886, 0.1500, 0.0000, 0.3142,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.40

In [ ]:
# 모듈을 순서대로 한 번에 통과시켜주는 클래스
seq_modules = nn.Sequential(
  flatten,
  layer1,
  nn.ReLU(),
  nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

# 1번째 축 방향으로 정규화 진행
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

print(pred_probab)

torch.Size([3, 10])
tensor([[0.1363, 0.0817, 0.1177, 0.1167, 0.1000, 0.0895, 0.0960, 0.0694, 0.1004,
         0.0923],
        [0.1445, 0.0700, 0.1206, 0.1096, 0.1065, 0.0806, 0.0972, 0.0700, 0.1077,
         0.0933],
        [0.1175, 0.0719, 0.1325, 0.1035, 0.1015, 0.0946, 0.1177, 0.0714, 0.0880,
         0.1015]], grad_fn=<SoftmaxBackward0>)
